# IPR-resolved stability of the nine square-QDM $(0,4)$ cages

This focused notebook rotates the degenerate cage eigenspace with the IPR strategy, classifies each preferred localized representative by its reduced-IZ mechanism, and compares its formal and exact local-perturbation compatibility spaces.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    summarize_cage_record_stability,
)
from qlinks.models import SquareQDMModel

## Build the model and choose a deterministic IPR basis

The random seed fixes the ordering for reproducibility. Record indices are not invariant under a different IPR seed, so the collective-cancellation record is identified again from its classification report.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=1.0e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()

square_search.counts_by_signature

In [ ]:
local_operators = square_build.kinetic_operators + square_build.potential_operators
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
local_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in local_operators
)
square_basis_configs = basis_configs_from_build_result(square_build)

## Classify and compare all nine records

`formal_compatible_dimension` counts first-order solvable directions. `exact_fixed_state_dimension` counts local coefficient directions that preserve the selected cage vector exactly. Their difference is the tangent-only, non-integrable sector.

In [ ]:
records = tuple(square_search[(0, 4)])
classifications = tuple(
    classify_cage_state(
        record.cage_state,
        kinetic_matrix=square_build.kinetic,
        basis_configs=square_basis_configs,
        hilbert_size=square_search.hilbert_size,
        sector_mask=None,
        config=CageClassificationConfig(
            amplitude_tolerance=1.0e-10,
            cancellation_tolerance=1.0e-9,
            action_tolerance=1.0e-9,
            sector_policy="infer_support_component",
            collective_cancellation_mode="all_problematic_nullspace",
        ),
    )
    for record in records
)

summaries = summarize_cage_record_stability(
    square_build.hamiltonian,
    local_term_matrices,
    records,
    classification_reports=classifications,
    coefficient_field="real",
    tolerance=1.0e-10,
)
record_table = pd.DataFrame(summary.to_summary_dict() for summary in summaries)
record_table

In [ ]:
individual = record_table[~record_table["requires_collective_cancellation"]]
collective = record_table[record_table["requires_collective_cancellation"]]

comparison = pd.DataFrame(
    [
        {
            "group": "individual reduced-IZ closure",
            "n_records": len(individual),
            "support_sizes": tuple(individual["support_size"]),
            "mean_ipr": individual["inverse_participation_ratio"].mean(),
            "formal_dimension": tuple(individual["formal_compatible_dimension"]),
            "exact_dimension": tuple(individual["exact_fixed_state_dimension"]),
            "tangent_only_dimension": tuple(individual["tangent_only_dimension"]),
        },
        {
            "group": "collective reduced-IZ cancellation",
            "n_records": len(collective),
            "support_sizes": tuple(collective["support_size"]),
            "mean_ipr": collective["inverse_participation_ratio"].mean(),
            "formal_dimension": tuple(collective["formal_compatible_dimension"]),
            "exact_dimension": tuple(collective["exact_fixed_state_dimension"]),
            "tangent_only_dimension": tuple(collective["tangent_only_dimension"]),
        },
    ]
)
comparison

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    record_table["inverse_participation_ratio"],
    record_table["exact_fixed_state_dimension"],
    s=70,
)
for _, row in record_table.iterrows():
    plt.annotate(
        str(int(row["record_index"])),
        (row["inverse_participation_ratio"], row["exact_fixed_state_dimension"]),
        xytext=(5, 4),
        textcoords="offset points",
    )
plt.xlabel("IPR of preferred cage representative")
plt.ylabel("exact cage-preserving coefficient dimension")
plt.title("Collective cancellation correlates with reduced local robustness")
plt.tight_layout()
plt.show()

## Interpretation

For this deterministic IPR basis, eight records are compact four-configuration cages. They require no collective reduced-IZ cancellation and every one has 44 exact local compatibility directions with no tangent-only directions. The remaining extended record has support 48, requires collective cancellation, and retains only 11 exact directions; four additional formal directions fail beyond first order.

This is record-level evidence, not yet a basis-independent theorem. The full nine-dimensional cage projector must still be tested separately to determine whether a perturbation destroys the manifold or merely rotates the preferred IPR representatives inside it.